In [2]:
from langgraph.graph import StateGraph , START , END
from dotenv import load_dotenv
from typing import TypedDict


In [3]:
class BatsmanState(TypedDict):
    
    runs : int 
    balls : int 
    fours : int
    sixes : int

    sr : float
    bpb : float 
    boundary_pct: float

    summary : str

In [4]:
def cal_sr(state : BatsmanState):

    balls = state['balls']
    runs = state['runs']

    sr = (runs/balls)*100

    return {'sr' : sr}


def cal_bpb(state:BatsmanState):

    fours = state['fours']
    sixes = state['sixes']

    balls = state['balls']

    bpb = balls/(fours+sixes)

    return {'bpb' : bpb}


def boundary_pct(state:BatsmanState):

    fours = state['fours']
    sixes = state['sixes']
    runs = state['runs']

    bdry_runs = (4*fours) + (6*sixes)

    boundary_pct = (bdry_runs/runs)*100

    return{'boundary_pct' : boundary_pct}


def summary(state:BatsmanState) :

    summary = f"""
Strike Rate = {state['sr']} \n
Boundaries per ball = {state['bpb']}\n
Boundary Percentage = {state['boundary_pct']}"""
    
    return {'summary' : summary}
    

    
    

# Linearly Nodes Adding 

In [5]:
graph = StateGraph(BatsmanState)


In [6]:
# Linearly 

graph.add_node('sr' , cal_sr)
graph.add_node('bpb' , cal_bpb)
graph.add_node('boundary_pct' , boundary_pct)
graph.add_node('summary' , summary)



In [7]:
#Add Edges

graph.add_edge(START , 'sr')
graph.add_edge('sr' , 'bpb')
graph.add_edge('bpb' , 'boundary_pct')
graph.add_edge('boundary_pct' , 'summary')
graph.add_edge('summary' , END)

wf = graph.compile()

In [8]:
inp = {'runs' : 100, 'balls' : 50 , 'fours' : 8 , 'sixes' : 8}

out = wf.invoke(inp)

print(out)



{'runs': 100, 'balls': 50, 'fours': 8, 'sixes': 8, 'sr': 200.0, 'bpb': 3.125, 'boundary_pct': 80.0, 'summary': '\nStrike Rate = 200.0 \n\nBoundaries per ball = 3.125\n\nBoundary Percentage = 80.0'}


# Parallelly Node Adding

In [9]:
gf = StateGraph(BatsmanState)

gf.add_node('cal_sr' , cal_sr)
gf.add_node('cal_bpb' , cal_bpb)
gf.add_node('boundary_pct' , boundary_pct)
gf.add_node('summary' , summary)


In [10]:
gf.add_edge(START , 'cal_sr')
gf.add_edge(START , 'cal_bpb')
gf.add_edge(START , 'boundary_pct')

gf.add_edge('cal_sr' , 'summary')
gf.add_edge('cal_bpb' , 'summary')
gf.add_edge('boundary_pct' , 'summary')

gf.add_edge('summary' , END)

In [11]:
wfp = gf.compile()

In [12]:
input = {'runs': 100 , 'balls' : 50 , 'fours' : 8 , 'sixes': 8}

out = wfp.invoke(input)

print(out['summary'])


Strike Rate = 200.0 

Boundaries per ball = 3.125

Boundary Percentage = 80.0


# WorkFlow 

- Linear 

- Parallel